# 01 — Preparación del Dataset

Este notebook transforma el JSON del dataset WCS Camera Traps (formato COCO) al formato requerido por YOLO.

**¿Qué hace?**
1. Lee el JSON con metadatos e imágenes del dataset Latam
2. Filtra las 12 especies más relevantes para fauna de Costa Rica / Centroamérica
3. Selecciona hasta 300 imágenes por especie (total ~3,600 imágenes)
4. Descarga las imágenes desde el servidor público de LILA
5. Convierte las bounding boxes de formato COCO a formato YOLO
6. Divide el dataset en train/val/test (70/20/10)
7. Genera el archivo `data.yaml` que necesita Ultralytics

**Tiempo estimado:** 20-40 minutos (descarga de imágenes)

**Prerequisitos:**
- Archivo `wcs_20220205_bboxes_latam_animals.json` en la carpeta `data/`
- Conexión a internet
- `pip install requests tqdm opencv-python-headless`

> **Nota:** Este notebook está pensado para correr en **Google Colab con GPU** ya que la descarga es más rápida desde la nube.

## 0. Instalación de dependencias

In [ ]:
# Solo necesario si corrés en Colab
# Si ya los tenés instalados, podés saltar esta celda
!pip install requests tqdm opencv-python-headless -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuración

In [ ]:
import json
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import requests
from tqdm.notebook import tqdm

# ─── Rutas ────────────────────────────────────────────────────────────────────
# ⚠️ Ajustá DATA_ROOT a tu ruta en Drive si cambia
DATA_ROOT = Path("/content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data")

JSON_PATH   = DATA_ROOT / "wcs_20220205_bboxes_latam_animals.json"
DATASET_DIR = DATA_ROOT / "dataset"   # aquí se guardan imágenes y labels
YAML_PATH   = DATA_ROOT / "data.yaml"

# ─── Parámetros ───────────────────────────────────────────────────────────────
MAX_PER_CLASS   = 300    # máximo de imágenes por especie
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.20
# TEST_RATIO    = 0.10  (lo que sobre)
RANDOM_SEED     = 42

# URL base para descargar imágenes (Azure — más estable)
BASE_URL = "https://lilawildlife.blob.core.windows.net/lila-wildlife/wcs-unzipped"
# Alternativa AWS si Azure falla:
# BASE_URL = "http://us-west-2.opendata.source.coop.s3.amazonaws.com/agentmorris/lila-wildlife/wcs-unzipped"

random.seed(RANDOM_SEED)

print("✅ Configuración lista")
print(f"   JSON      : {JSON_PATH}")
print(f"   Dataset   : {DATASET_DIR}")
print(f"   Max/clase : {MAX_PER_CLASS} imágenes")

✅ Configuración lista
   JSON      : /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/wcs_20220205_bboxes_latam_animals.json
   Dataset   : /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/dataset
   Max/clase : 300 imágenes


## 2. Cargar y explorar el JSON

In [ ]:
print("Cargando JSON... (puede tardar unos segundos)")
with open(JSON_PATH, "r") as f:
    coco = json.load(f)

print(f"✅ JSON cargado")
print(f"   Imágenes     : {len(coco['images']):,}")
print(f"   Anotaciones  : {len(coco['annotations']):,}")
print(f"   Categorías   : {len(coco['categories']):,}")

Cargando JSON... (puede tardar unos segundos)
✅ JSON cargado
   Imágenes     : 141,316
   Anotaciones  : 211,970
   Categorías   : 228


## 3. Selección de especies objetivo

Usamos las 12 especies más frecuentes en Guatemala (el país del dataset más similar a la fauna de Costa Rica).

In [ ]:
# Especies objetivo — fauna de Costa Rica / Centroamérica
TARGET_SPECIES = [
    "tayassu pecari",       # Chancho de monte / Quenk
    "crax rubra",           # Pavón / Great Curassow
    "leopardus pardalis",   # Ocelote
    "dasyprocta punctata",  # Guatusa / Central American Agouti
    "mazama temama",        # Cabro de monte / Brocket deer
    "puma concolor",        # Puma
    "tapirus bairdii",      # Danta / Baird's Tapir
    "panthera onca",        # Jaguar
    "nasua narica",         # Pizote / White-nosed Coati
    "odocoileus virginianus",# Venado cola blanca / White-tailed Deer
    "pecari tajacu",        # Saíno / Collared Peccary
    "meleagris ocellata",   # Pavo ocelado / Ocellated Turkey
]

# Diccionarios de mapeo
name_to_cat_id = {c["name"]: c["id"] for c in coco["categories"]}
cat_id_to_name = {c["id"]: c["name"]  for c in coco["categories"]}

# Verificar que todas las especies existen en el dataset
missing = [s for s in TARGET_SPECIES if s not in name_to_cat_id]
if missing:
    print(f"⚠️  Especies no encontradas en el dataset: {missing}")
else:
    print("✅ Todas las especies objetivo están en el dataset")

# Mapeo COCO category_id → YOLO class index (0-based, orden alfabético)
SORTED_CLASSES = sorted(TARGET_SPECIES)
YOLO_CLASS_NAMES = SORTED_CLASSES  # para data.yaml
class_to_yolo_idx = {name: i for i, name in enumerate(SORTED_CLASSES)}

target_cat_ids = {name_to_cat_id[s]: s for s in TARGET_SPECIES if s in name_to_cat_id}
cat_id_to_yolo = {cat_id: class_to_yolo_idx[name] for cat_id, name in target_cat_ids.items()}

print("\nMapeo de clases YOLO:")
for yolo_idx, name in enumerate(SORTED_CLASSES):
    cat_id = name_to_cat_id[name]
    print(f"  {yolo_idx:2d} → {name} (COCO id={cat_id})")

✅ Todas las especies objetivo están en el dataset

Mapeo de clases YOLO:
   0 → crax rubra (COCO id=374)
   1 → dasyprocta punctata (COCO id=3)
   2 → leopardus pardalis (COCO id=10)
   3 → mazama temama (COCO id=380)
   4 → meleagris ocellata (COCO id=372)
   5 → nasua narica (COCO id=240)
   6 → odocoileus virginianus (COCO id=378)
   7 → panthera onca (COCO id=24)
   8 → pecari tajacu (COCO id=8)
   9 → puma concolor (COCO id=6)
  10 → tapirus bairdii (COCO id=376)
  11 → tayassu pecari (COCO id=2)


## 4. Filtrar imágenes y anotaciones

In [ ]:
# Índices rápidos
img_country = {img["id"]: img["country_code"] for img in coco["images"]}
img_info    = {img["id"]: img for img in coco["images"]}

# Agrupar anotaciones por imagen (solo target species + GTM)
ann_by_img = defaultdict(list)
for a in coco["annotations"]:
    if a["category_id"] in target_cat_ids and a.get("bbox"):
        img_id = a["image_id"]
        if img_country.get(img_id) == "gtm":   # solo Guatemala
            ann_by_img[img_id].append(a)

print(f"Imágenes GTM con especies objetivo: {len(ann_by_img):,}")

# Agrupar imágenes por especie dominante
species_img_pool = defaultdict(list)
for img_id, anns in ann_by_img.items():
    cat_counts = Counter(a["category_id"] for a in anns)
    main_cat   = cat_counts.most_common(1)[0][0]
    species_img_pool[main_cat].append(img_id)

# Muestrear hasta MAX_PER_CLASS por especie
selected_img_ids = set()
print("\nImágenes seleccionadas por especie:")
for cat_id, img_ids in sorted(species_img_pool.items(), key=lambda x: target_cat_ids[x[0]]):
    sampled = random.sample(img_ids, min(MAX_PER_CLASS, len(img_ids)))
    selected_img_ids.update(sampled)
    print(f"  {target_cat_ids[cat_id]:35s} : {len(sampled):3d} imágenes (disponibles: {len(img_ids)})")

print(f"\n✅ Total imágenes seleccionadas : {len(selected_img_ids):,}")

Imágenes GTM con especies objetivo: 39,902

Imágenes seleccionadas por especie:
  crax rubra                          : 300 imágenes (disponibles: 8729)
  dasyprocta punctata                 : 300 imágenes (disponibles: 2712)
  leopardus pardalis                  : 300 imágenes (disponibles: 3249)
  mazama temama                       : 300 imágenes (disponibles: 2202)
  meleagris ocellata                  : 300 imágenes (disponibles: 11590)
  nasua narica                        : 300 imágenes (disponibles: 729)
  odocoileus virginianus              : 300 imágenes (disponibles: 813)
  panthera onca                       : 300 imágenes (disponibles: 1805)
  pecari tajacu                       : 300 imágenes (disponibles: 807)
  puma concolor                       : 300 imágenes (disponibles: 2031)
  tapirus bairdii                     : 300 imágenes (disponibles: 1780)
  tayassu pecari                      : 300 imágenes (disponibles: 3455)

✅ Total imágenes seleccionadas : 3,600


## 5. Dividir en train / val / test

In [ ]:
all_ids = list(selected_img_ids)
random.shuffle(all_ids)

n_total = len(all_ids)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)

train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train:n_train + n_val])
test_ids  = set(all_ids[n_train + n_val:])

splits = {"train": train_ids, "val": val_ids, "test": test_ids}

print("División del dataset:")
print(f"  Train : {len(train_ids):4d} imágenes  ({len(train_ids)/n_total*100:.0f}%)")
print(f"  Val   : {len(val_ids):4d} imágenes  ({len(val_ids)/n_total*100:.0f}%)")
print(f"  Test  : {len(test_ids):4d} imágenes  ({len(test_ids)/n_total*100:.0f}%)")

División del dataset:
  Train : 2520 imágenes  (70%)
  Val   :  720 imágenes  (20%)
  Test  :  360 imágenes  (10%)


## 6. Crear estructura de carpetas YOLO

In [ ]:
# Estructura esperada por Ultralytics:
# dataset/
#   train/images/  train/labels/
#   val/images/    val/labels/
#   test/images/   test/labels/

for split in ["train", "val", "test"]:
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"✅ Estructura de carpetas creada en {DATASET_DIR}")

✅ Estructura de carpetas creada en /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/dataset


## 7. Función de conversión COCO → YOLO

COCO usa `[x_min, y_min, width, height]` en píxeles absolutos.  
YOLO usa `[class x_center y_center width height]` normalizados a [0, 1].

In [ ]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    """
    Convierte bbox de formato COCO a formato YOLO.

    COCO : [x_min, y_min, width, height]  (píxeles absolutos)
    YOLO : [x_center, y_center, width, height]  (normalizados 0-1)
    """
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_w
    y_center = (y_min + h / 2) / img_h
    width    = w / img_w
    height   = h / img_h
    # Clamp a [0, 1] por si hay ruido en las anotaciones
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    width    = max(0.0, min(1.0, width))
    height   = max(0.0, min(1.0, height))
    return x_center, y_center, width, height


def download_image(file_name, dest_path, base_url=BASE_URL, timeout=20):
    """
    Descarga una imagen del servidor LILA.
    Retorna True si tuvo éxito, False si falló.
    """
    if dest_path.exists():
        return True  # ya descargada
    url = f"{base_url}/{file_name}"
    try:
        r = requests.get(url, timeout=timeout, stream=True)
        if r.status_code == 200:
            with open(dest_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
    except requests.RequestException:
        pass
    return False

print("✅ Funciones de conversión y descarga listas")

✅ Funciones de conversión y descarga listas


## 8. Descargar imágenes y generar labels YOLO

⏱️ Esta celda tarda entre **20 y 40 minutos** dependiendo de la conexión.  
Si se interrumpe, volvé a correrla — las imágenes ya descargadas se saltean.

In [ ]:
failed_downloads = []
stats = {"ok": 0, "skip": 0, "fail": 0}

for split_name, id_set in splits.items():
    print(f"\n📂 Procesando split: {split_name} ({len(id_set)} imágenes)")
    img_dir   = DATASET_DIR / split_name / "images"
    label_dir = DATASET_DIR / split_name / "labels"

    for img_id in tqdm(id_set, desc=split_name):
        img_meta = img_info[img_id]
        file_name = img_meta["file_name"]   # e.g. "animals/0399/1277.jpg"
        # Usar el id corto como nombre de archivo para evitar colisiones
        safe_name  = img_id.replace("-", "")[:16]
        img_path   = img_dir   / f"{safe_name}.jpg"
        label_path = label_dir / f"{safe_name}.txt"

        # 1. Descargar imagen
        ok = download_image(file_name, img_path)
        if not ok:
            failed_downloads.append(file_name)
            stats["fail"] += 1
            continue

        # 2. Verificar que la imagen sea legible con OpenCV
        frame = cv2.imread(str(img_path))
        if frame is None:
            img_path.unlink(missing_ok=True)
            failed_downloads.append(file_name)
            stats["fail"] += 1
            continue

        # 3. Generar label YOLO
        if not label_path.exists():
            img_h_real, img_w_real = frame.shape[:2]   # dimensiones reales vía OpenCV
            # Usar dimensiones reales si difieren del JSON (más robusto)
            img_w = img_w_real if img_w_real > 0 else img_meta["width"]
            img_h = img_h_real if img_h_real > 0 else img_meta["height"]
            anns  = ann_by_img.get(img_id, [])
            lines = []
            for a in anns:
                if a["category_id"] not in cat_id_to_yolo:
                    continue
                yolo_cls = cat_id_to_yolo[a["category_id"]]
                xc, yc, w, h = coco_bbox_to_yolo(a["bbox"], img_w, img_h)
                lines.append(f"{yolo_cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
            if lines:
                with open(label_path, "w") as f:
                    f.write("\n".join(lines))
                stats["ok"] += 1
            else:
                # Sin anotaciones válidas → borrar imagen descargada
                img_path.unlink(missing_ok=True)
                stats["skip"] += 1
        else:
            stats["ok"] += 1

print("\n" + "="*50)
print(f"✅ Procesadas   : {stats['ok']:,}")
print(f"⚠️  Sin labels   : {stats['skip']:,}")
print(f"❌ Falló descarga: {stats['fail']:,}")
if failed_downloads:
    print(f"\nPrimeras 5 fallas: {failed_downloads[:5]}")


📂 Procesando split: train (2520 imágenes)


train:   0%|          | 0/2520 [00:00<?, ?it/s]


📂 Procesando split: val (720 imágenes)


val:   0%|          | 0/720 [00:00<?, ?it/s]


📂 Procesando split: test (360 imágenes)


test:   0%|          | 0/360 [00:00<?, ?it/s]


✅ Procesadas   : 3,564
⚠️  Sin labels   : 0
❌ Falló descarga: 36

Primeras 5 fallas: ['animals/0574/1943.jpg', 'animals/0397/1229.jpg', 'animals/0397/1616.jpg', 'animals/0666/0408.jpg', 'animals/0379/0071.jpg']


## 9. Generar data.yaml

Este archivo le dice a Ultralytics dónde están los datos y cuáles son las clases.

In [ ]:
import yaml

# Nombres "amigables" para mostrar en las métricas y el video
COMMON_NAMES = {
    "crax rubra"              : "Pavon",
    "dasyprocta punctata"     : "Guatusa",
    "leopardus pardalis"      : "Ocelote",
    "mazama temama"           : "Cabro de monte",
    "meleagris ocellata"      : "Pavo ocelado",
    "nasua narica"            : "Pizote",
    "odocoileus virginianus"  : "Venado",
    "panthera onca"           : "Jaguar",
    "pecari tajacu"           : "Saino",
    "puma concolor"           : "Puma",
    "tapirus bairdii"         : "Danta",
    "tayassu pecari"          : "Chancho de monte",
}

yaml_content = {
    "path" : str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val"  : "val/images",
    "test" : "test/images",
    "nc"   : len(SORTED_CLASSES),
    "names": [COMMON_NAMES[n] for n in SORTED_CLASSES],
}

with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print(f"✅ data.yaml generado en {YAML_PATH}")
print("\nContenido:")
with open(YAML_PATH) as f:
    print(f.read())

✅ data.yaml generado en /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/data.yaml

Contenido:
names:
- Pavon
- Guatusa
- Ocelote
- Cabro de monte
- Pavo ocelado
- Pizote
- Venado
- Jaguar
- Saino
- Puma
- Danta
- Chancho de monte
nc: 12
path: /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/dataset
test: test/images
train: train/images
val: val/images



## 10. Verificación final

In [ ]:
print("=" * 55)
print("VERIFICACIÓN FINAL DEL DATASET")
print("=" * 55)
total_imgs = 0
for split in ["train", "val", "test"]:
    imgs   = list((DATASET_DIR / split / "images").glob("*.jpg"))
    labels = list((DATASET_DIR / split / "labels").glob("*.txt"))
    print(f"  {split:6s} — imágenes: {len(imgs):4d}  |  labels: {len(labels):4d}")
    total_imgs += len(imgs)

print(f"\n  TOTAL imágenes: {total_imgs:,}")
print(f"  Clases        : {len(SORTED_CLASSES)}")
print(f"  data.yaml     : {YAML_PATH}")
print()
print("✅ Dataset listo para entrenamiento.")
print("   Siguiente paso → abrir 02_training.ipynb en Colab con GPU")

VERIFICACIÓN FINAL DEL DATASET
  train  — imágenes: 2497  |  labels: 2497
  val    — imágenes:  710  |  labels:  710
  test   — imágenes:  357  |  labels:  357

  TOTAL imágenes: 3,564
  Clases        : 12
  data.yaml     : /content/drive/MyDrive/Wildlife Camera Trap Detector/wildlife-detector/data/data.yaml

✅ Dataset listo para entrenamiento.
   Siguiente paso → abrir 02_training.ipynb en Colab con GPU
